In [ ]:
import json
import re
import os
from pathlib import Path
import pandas as pd

RESULT_FOLDER_PATH = "/Users/lockewang/FIG/WebDomainRandomizer/baseline_results_new/baseline_results"
pd.set_option('display.max_colwidth', None)

# Loading predictions

In [ ]:
# load predictions from jsonl files for each model, reasoning_type, query_type combo into one dataframe

# Find all .jsonl files in the result folder
jsonl_files = list(Path(RESULT_FOLDER_PATH).glob("*.jsonl"))
print(f"Found {len(jsonl_files)} .jsonl files")

# Load all jsonl files into a list of dataframes
dataframes = []
for jsonl_file in jsonl_files:
    print(f"Loading {jsonl_file.name}")
    df = pd.read_json(jsonl_file, lines=True)
    dataframes.append(df)

# Concatenate all dataframes into one
df_all = pd.concat(dataframes, ignore_index=True)

print(f"\nTotal rows loaded: {len(df_all)}")
print(f"\nColumns: {df_all.columns.tolist()}")
print(f"\nFirst few rows:")
df_all.head()


Found 12 .jsonl files
Loading predictions_gta1_reasoning_relational_query_20251213_030654.jsonl
Loading predictions_uitars15_no_reasoning_relational_query_20251213_021703.jsonl
Loading predictions_qwen25vl_reasoning_relational_query_20251213_024630.jsonl
Loading predictions_uitars15_reasoning_direct_query_20251213_022342.jsonl
Loading predictions_gta1_no_reasoning_direct_query_20251213_030657.jsonl
Loading predictions_uitars15_no_reasoning_direct_query_20251213_021615.jsonl
Loading predictions_gta1_no_reasoning_relational_query_20251213_030656.jsonl
Loading predictions_qwen25vl_no_reasoning_relational_query_20251213_024625.jsonl
Loading predictions_qwen25vl_reasoning_direct_query_20251213_024628.jsonl
Loading predictions_uitars15_reasoning_relational_query_20251213_022352.jsonl
Loading predictions_qwen25vl_no_reasoning_direct_query_20251213_024623.jsonl
Loading predictions_gta1_reasoning_direct_query_20251213_030658.jsonl

Total rows loaded: 2400

Columns: ['model', 'use_reasoning', 'q

,model,use_reasoning,query_type,test_split,variant,task_id,step_index,instruction,raw_prediction,ground_truth_bbox,image_path
0,gta1,True,relational_query,test_task_style,original,012446b3-ee30-480b-86ec-3a3cdeaba9dc,3,Click on the button above 'May 17th 2023',Thought: I noticed that there is a gray heart-...,"[657, 300, 44, 42]",/mnt/test_splits/run_20251112_005055_test_task...
1,gta1,True,relational_query,test_task_style,precision,012446b3-ee30-480b-86ec-3a3cdeaba9dc,3,Click on the button above 'May 17th 2023',Thought: I noticed that there is a button labe...,"[747.8953247070312, 210, 30.79998779296875, 29...",/mnt/test_splits/run_20251112_005206_test_task...
2,gta1,True,relational_query,test_task_style,style,012446b3-ee30-480b-86ec-3a3cdeaba9dc,3,Click on the button above 'May 17th 2023',Thought: I noticed that there is a button labe...,"[657, 320.71875, 44, 42]",/mnt/test_splits/run_20251112_005316_test_task...
3,gta1,True,relational_query,test_task_style,text_shrink,012446b3-ee30-480b-86ec-3a3cdeaba9dc,3,Click on the button above 'May 17th 2023',Thought: I noticed that there is a button labe...,"[649.5, 300, 44, 42]",/mnt/test_splits/run_20251113_005227_test_task...
4,gta1,True,relational_query,test_task_style,original,012446b3-ee30-480b-86ec-3a3cdeaba9dc,5,Click on the button to the left of 'Dance',"Thought: I noticed that there is a ""Drinks"" op...","[1065.03125, 531.5, 53.28125, 17]",/mnt/test_splits/run_20251112_005055_test_task...


# Parse raw predictions into actions for each model

## Common helper code

In [20]:
import math

IMAGE_FACTOR = 28
MIN_PIXELS = 100 * 28 * 28
MAX_PIXELS = 16384 * 28 * 28
MAX_RATIO = 200

def escape_single_quotes(text):
    # 匹配未转义的单引号（不匹配 \\'）
    pattern = r"(?<!\\)'"
    return re.sub(pattern, r"\\'", text)

def round_by_factor(number: int, factor: int) -> int:
    """Returns the closest integer to 'number' that is divisible by 'factor'."""
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    """Returns the smallest integer greater than or equal to 'number' that is divisible by 'factor'."""
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    """Returns the largest integer less than or equal to 'number' that is divisible by 'factor'."""
    return math.floor(number / factor) * factor    


def smart_resize(
    height: int, width: int, factor: int = IMAGE_FACTOR, min_pixels: int = MIN_PIXELS, max_pixels: int = MAX_PIXELS
) -> tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:

    1. Both dimensions (height and width) are divisible by 'factor'.

    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].

    3. The aspect ratio of the image is maintained as closely as possible.
    """
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(
            f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
        )
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar


## Qwen2.5VL

Need to parse for number of <tool_call> </tool_tool>, each is an action. And then parse for arguments in
```
<tool_call>
{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [966, 546]}}
</tool_call>
```

In [67]:


# Parse action and visualize
def parse_qwen_action(response):
    resized_height, resized_width = smart_resize(
        1080,  # image.height,
        1920,  # image.width,
        factor=14 * 2,
        min_pixels=28 * 28,
        # max_pixels=self.processor.image_processor.max_pixels,
        max_pixels=MAX_PIXELS,
    )

    guide_text = "<tool_call>\n{\"name\": \"computer_use\", \"arguments\": {\"action\": \"left_click\", \"coordinate\": ["
    response = guide_text + response
    cut_index = response.rfind('}')
    if cut_index != -1:
        response = response[:cut_index + 1]
    try:
        result_dict = {
            "result": "positive",
            "format": "x1y1x2y2",
            "raw_response": response,
            "bbox": None,
            "point": None
        }

        action = json.loads(response.split('<tool_call>\n')[1].split('\n</tool_call>')[0])
        coordinates = action['arguments']['coordinate']
        if len(coordinates) == 2:
            point_x, point_y = coordinates
        elif len(coordinates) == 4:
            x1, y1, x2, y2 = coordinates
            point_x = (x1 + x2) / 2
            point_y = (y1 + y2) / 2
        else:
            raise ValueError("Wrong output format")
        print(point_x, point_y)
        result_dict["point"] = [point_x / resized_width, point_y / resized_height]  # Normalize predicted coordinates
    except (IndexError, KeyError, TypeError, ValueError) as e:
        pass
    
    return result_dict

In [ ]:
qwen_filter = df_all['model'] == 'qwen25vl'
qwen_df = df_all[qwen_filter].copy()
qwen_df['structured_actions'] = qwen_df['raw_prediction'].apply(parse_qwen_action)
qwen_df['structured_actions'].head()
qwen_df['structured_actions']


400                                                                                                             {'result': 'positive', 'format': 'x1y1x2y2', 'raw_response': '<tool_call>
{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [Thought: I need to click on the button labeled 'May 17th 2023' to select this date for the cooking experience.
<tool_call>
{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [684, 375]}}', 'bbox': None, 'point': None}
401    {'result': 'positive', 'format': 'x1y1x2y2', 'raw_response': '<tool_call>
{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [Thought: I need to select a date for the cooking experience. The calendar shows May 17th 2023, which is highlighted, indicating it might be the selected date. Clicking on this date will likely confirm the selection.
<tool_call>
{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [768, 258]}}', 'bbox': None,

## UI-TARS1.5

Parse for `Action: click(start_box='(681,385)')`

In [37]:
import ast

# UI-TARS1.5 action parsing functions
def parse_action(action_str):
    try:
        # 解析字符串为 AST 节点
        node = ast.parse(action_str, mode='eval')

        # 确保节点是一个表达式
        if not isinstance(node, ast.Expression):
            raise ValueError("Not an expression")

        # 获取表达式的主体
        call = node.body

        # 确保主体是一个函数调用
        if not isinstance(call, ast.Call):
            raise ValueError("Not a function call")

        # 获取函数名
        if isinstance(call.func, ast.Name):
            func_name = call.func.id
        elif isinstance(call.func, ast.Attribute):
            func_name = call.func.attr
        else:
            func_name = None

        # 获取关键字参数
        kwargs = {}
        for kw in call.keywords:
            key = kw.arg
            # 处理不同类型的值，这里假设都是常量
            if isinstance(kw.value, ast.Constant):
                value = kw.value.value
            elif isinstance(kw.value, ast.Str):  # 兼容旧版本 Python
                value = kw.value.s
            else:
                value = None
            kwargs[key] = value

        return {
            'function': func_name,
            'args': kwargs
        }

    except Exception as e:
        print(f"Failed to parse action '{action_str}': {e}")
        return None

def parse_action_to_structure_output(text, factor, origin_resized_height, origin_resized_width, model_type, max_pixels=16384*28*28, min_pixels=100*28*28):
    text = text.strip()
    if model_type == "qwen25vl":
        smart_resize_height, smart_resize_width = smart_resize(origin_resized_height, origin_resized_width, factor=IMAGE_FACTOR, min_pixels=min_pixels, max_pixels=max_pixels)

    # 正则表达式匹配 Action 字符串
    if text.startswith("Thought:"):
        thought_pattern = r"Thought: (.+?)(?=\s*Action:|$)"
        thought_hint = "Thought: "
    elif text.startswith("Reflection:"):
        thought_pattern = r"Reflection: (.+?)Action_Summary: (.+?)(?=\s*Action:|$)"
        thought_hint = "Reflection: "
    elif text.startswith("Action_Summary:"):
        thought_pattern = r"Action_Summary: (.+?)(?=\s*Action:|$)"
        thought_hint = "Action_Summary: "
    else:
        thought_pattern = r"Thought: (.+?)(?=\s*Action:|$)"
        thought_hint = "Thought: "
    reflection, thought = None, None
    thought_match = re.search(thought_pattern, text, re.DOTALL)
    if thought_match:
        if len(thought_match.groups()) == 1:
            thought = thought_match.group(1).strip()
        elif len(thought_match.groups()) == 2:
            thought = thought_match.group(2).strip()
            reflection = thought_match.group(1).strip()
    assert "Action:" in text
    action_str = text.split("Action:")[-1]

    tmp_all_action = action_str.split("\n\n")
    all_action = []
    for action_str in tmp_all_action:
        if "type(content" in action_str:
            # 正则表达式匹配 content 中的字符串并转义单引号
            def escape_quotes(match):
                content = match.group(1)  # 获取 content 的值
                return content

            # 使用正则表达式进行替换
            pattern = r"type\(content='(.*?)'\)"  # 匹配 type(content='...')
            content = re.sub(pattern, escape_quotes, action_str)

            # 处理字符串
            action_str = escape_single_quotes(content)
            action_str = "type(content='" + action_str + "')"
        all_action.append(action_str)

    parsed_actions = [parse_action(action.replace("\n","\\n").lstrip()) for action in all_action]
    actions = []
    for action_instance, raw_str in zip(parsed_actions, all_action):
        if action_instance == None:
            print(f"Action can't parse: {raw_str}")
            raise ValueError(f"Action can't parse: {raw_str}") 
        action_type = action_instance["function"]
        params = action_instance["args"]

        # import pdb; pdb.set_trace()
        action_inputs = {}
        for param_name, param in params.items():
            if param == "": continue
            param = param.lstrip()  # 去掉引号和多余的空格
            # 处理start_box或者end_box参数格式 '<bbox>x1 y1 x2 y2</bbox>'
            action_inputs[param_name.strip()] = param
            
            if "start_box" in param_name or "end_box" in param_name:
                ori_box = param
                # Remove parentheses and split the string by commas
                numbers = ori_box.replace("(", "").replace(")", "").split(",")

                # Convert to float and scale by 1000
                # Qwen2.5vl output absolute coordinates, qwen2vl output relative coordinates
                if model_type == "qwen25vl":
                    float_numbers = []
                    for num_idx, num in enumerate(numbers):
                        num = float(num)
                        if (num_idx + 1) % 2 == 0:
                            float_numbers.append(float(num/smart_resize_height))
                        else:
                            float_numbers.append(float(num/smart_resize_width))
                else:
                    float_numbers = [float(num) / factor for num in numbers]

                if len(float_numbers) == 2:
                    float_numbers = [float_numbers[0], float_numbers[1], float_numbers[0], float_numbers[1]]
                action_inputs[param_name.strip()] = str(float_numbers)

        # import pdb; pdb.set_trace()
        actions.append({
            "reflection": reflection,
            "thought": thought,
            "action_type": action_type,
            "action_inputs": action_inputs,
            "text": text
        })
    return actions


In [35]:
uitars15_filter = df_all['model'] == 'uitars15'
uitars15_df = df_all[uitars15_filter].copy()
uitars15_df['raw_prediction']


200                  Action: click(start_box='(258,414)')
201                  Action: click(start_box='(471,291)')
202                  Action: click(start_box='(260,435)')
203                  Action: click(start_box='(1352,41)')
204                 Action: click(start_box='(1190,548)')
                              ...                        
1995    Thought: I noticed a prominent red button in t...
1996    Thought: I noticed that a notification prompt ...
1997    Thought: I noticed a pop-up notification windo...
1998    Thought: I noticed that a notification prompt ...
1999    Thought: I noticed a notification pop-up on th...
Name: raw_prediction, Length: 800, dtype: object

In [66]:
from pprint import pprint

uitars15_df['structured_actions'] = uitars15_df['raw_prediction'].apply(parse_action_to_structure_output, factor=IMAGE_FACTOR, origin_resized_height=1920, origin_resized_width=1080, model_type="qwen25vl")  # use qwen25vl for uitars15
pd.set_option('display.max_columns', None)

def has_more_than_1_action(row):
    return True if len(row['structured_actions']) > 1 else False

print("rows with more than 1 action:")
display(len(uitars15_df[uitars15_df.apply(has_more_than_1_action, axis=1)]))


def has_non_click_action(row):
    return True if any(action['action_type'] != 'click' for action in row['structured_actions']) else False

display(uitars15_df[uitars15_df.apply(has_non_click_action, axis=1)][['raw_prediction', 'structured_actions']].head())


rows with more than 1 action:


0

,raw_prediction,structured_actions
1825,"Thought: I noticed that the page is currently displaying the list of ingredients arranged in alphabetical order. To find the button under ""KITCHEN TIPS,"" I need to scroll down the page. Let me use the mouse wheel to scroll down and see what content is available below.\nAction: scroll(direction='down', start_box='(1355,537)')","[{'reflection': None, 'thought': 'I noticed that the page is currently displaying the list of ingredients arranged in alphabetical order. To find the button under ""KITCHEN TIPS,"" I need to scroll down the page. Let me use the mouse wheel to scroll down and see what content is available below.', 'action_type': 'scroll', 'action_inputs': {'direction': 'down', 'start_box': '[1.2408424908424909, 0.2779503105590062, 1.2408424908424909, 0.2779503105590062]'}, 'text': 'Thought: I noticed that the page is currently displaying the list of ingredients arranged in alphabetical order. To find the button under ""KITCHEN TIPS,"" I need to scroll down the page. Let me use the mouse wheel to scroll down and see what content is available below. Action: scroll(direction='down', start_box='(1355,537)')'}]"
1961,"Thought: I noticed that the page is still loading, and the content hasn't fully appeared yet. I need to wait a moment for the page to finish loading before I can proceed with the next steps.\nAction: wait()","[{'reflection': None, 'thought': 'I noticed that the page is still loading, and the content hasn't fully appeared yet. I need to wait a moment for the page to finish loading before I can proceed with the next steps.', 'action_type': 'wait', 'action_inputs': {}, 'text': 'Thought: I noticed that the page is still loading, and the content hasn't fully appeared yet. I need to wait a moment for the page to finish loading before I can proceed with the next steps. Action: wait()'}]"


## GTA1

Parse for:
1. `Action: click(start_box='(648,409)')`
2. 
    ```
    Thought: I noticed a search box at the top of the page, which is perfect for looking up pork. I need to click on it first to make it active so that I can enter the keyword.
    Action: (583,287)
    ```
3. `(1444,42)`
4. `-203,240`
5. 
    ```
    Thought: I noticed the link for Audiology, which is located just below the Allergies & Immunology link. It's positioned in the left column, and I need to click on it to access the relevant page.
    Action: (389,546)
    assistant
    Thought: I'm currently looking at the list of medical specialties, and I've found the link for Anesthesiology. It's located in the left column, just below the Allergy & Immunology link. I need to click on it to learn more about this field.
    Action: (408,503)
    assistant
    Thought: I'm currently browsing through the list of medical specialties and I've come across the link for Audiology. It's positioned in the left column, just below the Allergy & Immunology link. I need to click on it to get more information about this field.
    Action: (389,546)
    assistant
    Thought: I just clicked on the link for Audiology, and now I need to move on to the next link. I see that the link for Anesthesiology is right below it, so I'll click on that one next. This way, I can continue exploring the various medical specialties.
    Action: (408,503)
    assistant
    Thought: I just clicked on the link for Audiology, and now I need to move on to the next link. I see that the link for Anesthesiology is right below it, so I'll click on that one next. This way, I can continue exploring the various medical specialties.
    Action: (408,503)
    assistant
    Thought: I just clicked on the link for Audiology, and now I need to move on to the next link. I see that the link for Anesthesiology is right below it, so I'll click on that one next. This way, I can continue exploring the various medical specialties.
    Action: (408,503)
    assistant
    ...
    ```

In [69]:
def extract_coordinates(raw_string):
    try:
        matches = re.findall(r"\((-?\d*\.?\d+),\s*(-?\d*\.?\d+)\)", raw_string)
        return [tuple(map(int, match)) for match in matches][0]
    except:
        return 0,0


In [ ]:
gta1_filter = df_all['model'] == 'gta1'
gta1_df = df_all[gta1_filter].copy()
gta1_df['coordinates'] = gta1_df['raw_prediction'].apply(extract_coordinates)
gta1_df['coordinates']

0    (1416, 317)
1     (739, 195)
2     (728, 345)
3     (834, 583)
4    (1266, 548)
Name: coordinates, dtype: object

# Combine per model df and save to csv for inspection

In [76]:
pd.concat([gta1_df, qwen_df, uitars15_df]).to_csv('baseline_results_combined.csv', index=False)

# Check for anomalies (na, malformed, etc)

# Denormalize all coordinates

# Calculate action str exact match

# Calculate hit box accuracy

# Calculate bbox center MSE & NSME

# Calculate GIoU & Normalized GIoU with adaptive proxy bbox for predicted 2D coordinates

# Plotting trends